In [1]:
# dialogue vs narrative/descriptive tex 

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from collections import Counter
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.util import ngrams
from nltk import FreqDist

/Users/art/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
[nltk_data] Downloading package punkt to /Users/art/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/art/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/art/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
with open('article.txt') as f:
    text = f.read()

pharagraps = [p.strip() for p in text.split("\n") if len(p.strip())> 80]
print(len(pharagraps))
for i , p in  enumerate(pharagraps[:3]):
    print(f'Phar {i+1} {len(p)}, {p[:100]}')

66
Phar 1 267, A Michigan surgeon invented an apparatus that he believes tricks the brain into thinking the stomach
Phar 2 399, Bonnie Lauria was miserable. She was subsisting on liquids and a handful of foods her stomach could 
Phar 3 419, During gastric bypass surgery, the stomach is reduced to about the size of a walnut and attached to 


In [4]:
def clean_tokenize(pharagraph):
    tokens = word_tokenize(pharagraph.lower())
    alist = [t for t in tokens if t.isalpha() and t not in stopwords.words('english')]
    return alist

In [5]:
tokenized_paragraphs = [clean_tokenize(p) for p in pharagraps]
tokenized_paragraphs[0][:5]

['michigan', 'surgeon', 'invented', 'apparatus', 'believes']

In [6]:
doc_freq = FreqDist()
for tokens in tokenized_paragraphs:
    for word in set(tokens):
        doc_freq[word] +=1
n_docs = len(pharagraps)
too_frequent = {w for w in doc_freq if doc_freq[w] / n_docs > 0.15}
sorted(too_frequent)

['baker',
 'bariatric',
 'could',
 'device',
 'food',
 'full',
 'like',
 'loss',
 'one',
 'patient',
 'patients',
 'says',
 'sense',
 'stomach',
 'surgery',
 'weight']

In [7]:
def nltk_tokenizer(pharagraph):
    tokens = word_tokenize(pharagraph.lower())
    alist = [t for t in tokens if t.isalpha() and t not in stopwords.words('english')]
    tokens = [t for t in alist if t not in too_frequent]
    return tokens

In [8]:
vectorizer = TfidfVectorizer(
    tokenizer=nltk_tokenizer,
    ngram_range=(1,2),
    min_df=2,
    max_features=100,
    lowercase=False
)

In [9]:
X = vectorizer.fit_transform(pharagraps)

/Users/art/anaconda3/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [10]:
X.shape

(66, 100)

In [11]:
def has_quote(p):
    return 'quote' if ('\u201c' in p or '\u201d' in p ) else 'no_quote'

In [12]:
raw_labels = [has_quote(p) for p in pharagraps]
encoder = LabelEncoder()
y = encoder.fit_transform(raw_labels)
y
# raw_labels

array([0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1,
       0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1,
       0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=42)

In [14]:
models = [
    ("Naive Bayes", MultinomialNB()),
    ("Logistic Regression", LogisticRegression()),
    ("KNN", KNeighborsClassifier()),
    ("Decision Tree", DecisionTreeClassifier()),
    ("Random Forest", RandomForestClassifier()),
    ("Linear Support Vector Classification", LinearSVC())
]

In [15]:
for model_name, model in models:
    model.fit(X_train,y_train)
    preds = model.predict(X_test)
    print(model_name, model.score(X_test,y_test))
    print(classification_report(y_test, preds, target_names=encoder.classes_))

Naive Bayes 0.6
              precision    recall  f1-score   support

    no_quote       0.00      0.00      0.00         8
       quote       0.60      1.00      0.75        12

    accuracy                           0.60        20
   macro avg       0.30      0.50      0.38        20
weighted avg       0.36      0.60      0.45        20

Logistic Regression 0.6
              precision    recall  f1-score   support

    no_quote       0.00      0.00      0.00         8
       quote       0.60      1.00      0.75        12

    accuracy                           0.60        20
   macro avg       0.30      0.50      0.38        20
weighted avg       0.36      0.60      0.45        20

KNN 0.65
              precision    recall  f1-score   support

    no_quote       1.00      0.12      0.22         8
       quote       0.63      1.00      0.77        12

    accuracy                           0.65        20
   macro avg       0.82      0.56      0.50        20
weighted avg       0.78  

/Users/art/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/art/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/art/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/